<a href="https://colab.research.google.com/github/Vishal-Kawade00/Agentic-Ai/blob/main/ai-engine/Hybrid_RAG_Inference_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# CELL 1: DEPENDENCIES & ENVIRONMENT SET UP
# ==========================================
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q pdfplumber sentence-transformers chromadb langchain-text-splitters rank_bm25

print("✅ Enterprise AI dependencies installed successfully.")

✅ Enterprise AI dependencies installed successfully.


In [2]:
# ==========================================
# CELL 3: VECTOR INTELLIGENCE & CHROMA DB
# ==========================================
import pdfplumber
import io
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the Embedding Model (Open-source, runs locally in Colab)
# "all-MiniLM-L6-v2" is incredibly fast and highly rated for standard RAG.
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 2. Initialize ChromaDB (Stateless / In-Memory configuration)
chroma_client = chromadb.Client()

# Reset collection if it exists (useful when you re-run the cell during testing)
try:
    chroma_client.delete_collection(name="rag_documents")
except:
    pass

vector_db = chroma_client.create_collection(name="rag_documents", embedding_function=emb_fn)

# 3. Text Extraction
def extract_text_with_metadata(pdf_bytes):
    pages_data = []
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text:
                pages_data.append({"page_number": i + 1, "text": text.replace('\n', ' ').strip()})
    return pages_data

# 4. Advanced Recursive Chunking & Database Ingestion
def process_and_store_document(pages_data, filename):
    """
    Chunks text safely and embeds it into ChromaDB.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,        # Size of the chunk
        chunk_overlap=150,     # Overlap to prevent losing context between chunks
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""] # The "Recursive" magic
    )

    documents = []
    metadatas = []
    ids = []

    chunk_counter = 0

    for page in pages_data:
        # Split the page text intelligently
        chunks = text_splitter.split_text(page["text"])

        for chunk in chunks:
            documents.append(chunk)
            # Tag every chunk with where it came from (Crucial for UI Citations!)
            metadatas.append({
                "source": filename,
                "page_number": page["page_number"]
            })
            ids.append(f"{filename}_chunk_{chunk_counter}")
            chunk_counter += 1

    # Insert into Vector Database
    if documents:
        vector_db.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )

    return chunk_counter

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
# ==========================================
# CELL 4: THE INGESTION ENDPOINT
# ==========================================
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import time

app = FastAPI(title="RAG Inference Engine")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.post("/process-pdf")
async def process_pdf(file: UploadFile = File(...)):
    try:
        start_time = time.time()
        contents = await file.read()

        print(f"[INGEST] Parsing & Embedding {file.filename}...")

        # 1. Extract Text
        pages_data = extract_text_with_metadata(contents)
        total_pages = len(pages_data)

        # 2. Chunk and Store in ChromaDB
        total_chunks = process_and_store_document(pages_data, file.filename)

        process_time = time.time() - start_time
        print(f"✅ [SUCCESS] Vectorized {total_pages} pages into {total_chunks} chunks.")

        return {
            "status": "success",
            "message": "Document embedded successfully into Vector Space.",
            "metadata": {
                "total_pages": total_pages,
                "total_vector_chunks": total_chunks,
                "processing_time_sec": round(process_time, 2)
            }
        }

    except Exception as e:
        print(f"❌ [ERROR] Processing failed: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

In [4]:
# ==========================================
# CELL 5: THE MASTER PIPELINE (HYBRID + RERANK)
# ==========================================
import google.generativeai as genai
from pydantic import BaseModel
from google.colab import userdata
from fastapi import HTTPException
import time
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# 1. Initialize Gemini
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    llm_model = genai.GenerativeModel('gemini-2.5-flash')
except Exception as e:
    print("❌ ERROR: Could not find GEMINI_API_KEY.")

# 2. Initialize the Cross-Encoder Judge
print("⚖️ Loading Cross-Encoder Judge Model...")
# This is a highly respected, lightweight reranking model
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
print("✅ Cross-Encoder initialized.")

# --- GLOBAL MEMORY FOR BM25 ---
global_bm25_corpus = []
global_bm25_metadata = []
global_bm25_index = None

class QueryRequest(BaseModel):
    query: str

# ==========================================
# SEARCH LOGIC
# ==========================================
def build_bm25_index():
    global global_bm25_corpus, global_bm25_metadata, global_bm25_index
    all_data = vector_db.get()
    if not all_data['documents']: return False
    global_bm25_corpus = all_data['documents']
    global_bm25_metadata = all_data['metadatas']
    tokenized_corpus = [doc.lower().split(" ") for doc in global_bm25_corpus]
    global_bm25_index = BM25Okapi(tokenized_corpus)
    return True

def reciprocal_rank_fusion(vector_results, bm25_results, k=60):
    fused_scores = {}
    for rank, doc_id in enumerate(vector_results):
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + 1 / (k + rank)
    for rank, doc_id in enumerate(bm25_results):
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + 1 / (k + rank)
    reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [item[0] for item in reranked]

def retrieve_and_rerank(query_text, initial_k=10, final_k=3):
    """
    1. Hybrid Search gets Top 10.
    2. Cross-Encoder reranks those 10.
    3. Returns Top 3 for the LLM.
    """
    if global_bm25_index is None: build_bm25_index()

    # --- PHASE 1: HYBRID RETRIEVAL ---
    v_results = vector_db.query(query_texts=[query_text], n_results=initial_k)
    vector_docs = v_results['documents'][0] if v_results['documents'] else []

    tokenized_query = query_text.lower().split(" ")
    bm25_scores = global_bm25_index.get_scores(tokenized_query)
    bm25_top_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:initial_k]
    bm25_docs = [global_bm25_corpus[i] for i in bm25_top_indices]

    fused_texts = reciprocal_rank_fusion(vector_docs, bm25_docs)[:initial_k]

    if not fused_texts: return [], []

    # --- PHASE 2: CROSS-ENCODER RERANKING ---
    # Create pairs of [Query, Document Chunk]
    cross_inp = [[query_text, text] for text in fused_texts]

    # Judge the pairs
    cross_scores = reranker_model.predict(cross_inp)

    # Sort by the new, highly accurate Cross-Encoder scores
    scored_docs = zip(cross_scores, fused_texts)
    sorted_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)

    # Extract the absolute best text chunks
    final_contexts = [doc for score, doc in sorted_docs[:final_k]]

    # Extract Citations
    citations = []
    for text in final_contexts:
        try:
            idx = global_bm25_corpus.index(text)
            meta = global_bm25_metadata[idx]
            citations.append({"source": meta['source'], "page": meta['page_number']})
        except ValueError:
            pass

    return final_contexts, citations

# ==========================================
# LLM GENERATION
# ==========================================
def generate_grounded_answer(query, contexts):
    context_str = "\n\n---\n\n".join(contexts)
    prompt = f"""
    You are an expert AI assistant for document analysis.
    Answer the user's question STRICTLY using the provided context below.
    If the answer cannot be found in the context, output exactly: "I cannot answer this based on the provided document."

    CONTEXT:
    {context_str}

    QUESTION:
    {query}

    ANSWER:
    """
    response = llm_model.generate_content(prompt)
    return response.text

# ==========================================
# THE CHAT ENDPOINT
# ==========================================
@app.post("/ask")
async def ask_question(request: QueryRequest):
    try:
        start_time = time.time()
        print(f"\n[MASTER PIPELINE] Query: '{request.query}'")

        contexts, citations = retrieve_and_rerank(request.query)

        if not contexts:
            return {"status": "success", "answer": "No relevant context found.", "citations": []}

        answer = generate_grounded_answer(request.query, contexts)

        process_time = time.time() - start_time
        print(f"✅ [SUCCESS] Reranked response generated in {process_time:.2f}s")

        return {
            "status": "success",
            "search_type": "Hybrid RRF + Cross-Encoder Reranking", # <--- HUGE RESUME FLEX
            "answer": answer,
            "citations": citations,
            "processing_time_sec": round(process_time, 2)
        }

    except Exception as e:
        print(f"❌ [ERROR] AI Generation failed: {str(e)}")
        raise HTTPException(status_code=500, detail="Failed to generate AI response")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


⚖️ Loading Cross-Encoder Judge Model...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Cross-Encoder initialized.


In [5]:
# ==========================================
# FINAL CELL: THE PURE ASGI SERVER
# ==========================================
from pyngrok import ngrok
import uvicorn
from google.colab import userdata
import asyncio

# 1. Securely fetch and set Ngrok Token
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except Exception as e:
    print("❌ ERROR: Could not find NGROK_TOKEN in Colab Secrets.")
    raise e

# 2. Open the Tunnel
port = 8000
public_url = ngrok.connect(port).public_url

print("\n" + "="*50)
print("="*50 + "\n")

# 3. Start the Server natively on the IPython Event Loop
config = uvicorn.Config(app, host="0.0.0.0", port=port)
server = uvicorn.Server(config)

# Note: This will block the cell from finishing. Click 'Stop' if you need to edit code.
await server.serve()

INFO:     Started server process [5447]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)




[INGEST] Parsing & Embedding DBMS_Full_Notes.pdf...
✅ [SUCCESS] Vectorized 49 pages into 150 chunks.
INFO:     2001:4490:4479:516c:55ce:d945:68a2:9f56:0 - "POST /process-pdf HTTP/1.1" 200 OK

[MASTER PIPELINE] Query: 'what is this pdf about'
✅ [SUCCESS] Reranked response generated in 3.38s
INFO:     2001:4490:4479:516c:55ce:d945:68a2:9f56:0 - "POST /ask HTTP/1.1" 200 OK

[MASTER PIPELINE] Query: 'what it contents list all contents'
✅ [SUCCESS] Reranked response generated in 5.29s
INFO:     2001:4490:4479:516c:55ce:d945:68a2:9f56:0 - "POST /ask HTTP/1.1" 200 OK

[MASTER PIPELINE] Query: 'what this pdf includes list all the topics it includes'
✅ [SUCCESS] Reranked response generated in 3.77s
INFO:     2001:4490:4479:516c:55ce:d945:68a2:9f56:0 - "POST /ask HTTP/1.1" 200 OK
[INGEST] Parsing & Embedding React & Redux Notes.pdf...
✅ [SUCCESS] Vectorized 94 pages into 96 chunks.
INFO:     2001:4490:4479:516c:55ce:d945:68a2:9f56:0 - "POST /process-pdf HTTP/1.1" 200 OK

[MASTER PIPELINE] Quer

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5447]
